In [ ]:
import requests
import zipfile
import torch
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

In [ ]:
url = "https://firebasestorage.googleapis.com/v0/b/grandmacan-2dae4.appspot.com/o/ML_data%2Fone_piece_full.zip?alt=media&token=937656fd-f5c1-44f5-b174-1e2d590b8ef3"

with open("one_piece_full.zip", "wb") as f:
    req = requests.get(url)
    f.write(req.content)

with zipfile.ZipFile("one_piece_full.zip", "r") as zip_file:
    zip_file.extractall("one_piece_full")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def accuracy_fn(y_pred, y_true):
    correct_num = (y_pred==y_true).sum()
    acc = correct_num / len(y_true) * 100
    return acc

def train_step(dataloader, model, cost_fn, optimizer, accuracy_fn, device):
    train_cost = 0
    train_acc = 0
    for batch, (x, y) in enumerate(dataloader):
        x = x.to(device)
        y = y.to(device)
        model.to(device)

    model.train()

    y_pred = model(x)

    cost = cost_fn(y_pred, y)

    train_cost += cost
    train_acc += accuracy_fn(y_pred.argmax(dim=1), y)

    optimizer.zero_grad()

    cost.backward()

    optimizer.step()

    train_cost /= len(train_dataloader)
    train_acc /= len(train_dataloader)

    print(f"\nTrain Cost: {train_cost:.4f}, Train Acc: {train_acc:.2f}")


def test_step(dataloader, model, cost_fn, accuracy_fn, device):
    test_cost = 0
    test_acc = 0
    model.eval()
    with torch.inference_mode():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)
            model.to(device)

        test_pred = model(x)

        test_cost += cost_fn(test_pred, y)
        test_acc += accuracy_fn(test_pred.argmax(dim=1), y)

    test_cost /= len(test_dataloader)
    test_acc /= len(test_dataloader)

    print(f"Test Cost: {test_cost:.4f}, Test Acc: {test_acc:.2f} \n")

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root, train, transform=None):
        if train:
            image_root = Path(root) / "train"
        else:
            image_root = Path(root) / "test"
        
        with open(Path(root) / "classnames.txt", "r") as f:
            lines = f.readlines()
            self.classes = [line.strip() for line in lines]
        self.paths = [i for i in image_root.rglob("*") if i.is_file()]
        self.transform = transform

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert('RGB')
        class_name = self.paths[index].parent.name
        class_index = self.classes.index(class_name)

        if self.transform:
            return self.transform(image), class_index
        else:
            return image, class_index
    def __len__(self):
        return len(self.paths)

In [ ]:
import torchvision
weights = torchvision.models.EfficientNet_B1_Weights.DEFAULT
model = torchvision.models.efficientnet_b1(weights=weights)
model.to(device)

In [ ]:
efficientnet_b1_transforms = weights.transforms()
efficientnet_b1_transforms

In [ ]:
train_dataset = ImageDataset(root="one_piece_full", 
                train=True, 
                transform=efficientnet_b1_transforms
)

test_dataset = ImageDataset(root="one_piece_full", 
                train=False, 
                transform=efficientnet_b1_transforms
)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_dataloader = DataLoader(dataset=train_dataset,
                batch_size=BATCH_SIZE,
                shuffle=True
)

test_dataloader = DataLoader(dataset=test_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False
)

In [ ]:
len(train_dataloader), len(test_dataloader)

In [ ]:
from torchinfo import summary

summary(model=model, input_size=[16, 3, 64, 64], col_names=['input_size', 'output_size', 'num_params', 'trainable'], row_settings=['var_names'])

In [ ]:
from torch import nn
model.classifier[1] = nn.Linear(in_features=1280, out_features=18, bias=True).to(device)

In [ ]:
for param in model.features.parameters():
    param.requires_grad=False
    print(param)

In [ ]:
cost_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

In [ ]:
epochs = 10

for epoch in range(epochs):
    print(f'Epoch: {epoch}\n-------------')

    train_step(train_dataloader, model, cost_fn, optimizer, accuracy_fn, device)
    test_step(test_dataloader, model, cost_fn, accuracy_fn, device)

In [ ]:
image = Image.open('luffy.png').convert('RGB')
image = efficientnet_b1_transforms(image)

model.eval()
with torch.inference_mode():
    y_pred = model(image.to(device))
y_pred = torch.softmax(y_pred)
class_idx = y_pred.argmax(dim=1)
test_dataset.classes[class_idx]